In [1]:
from datasets import load_dataset, load_from_disk
from replay.metrics import Recall, Precision, HitRate
import polars as pl
from scipy.sparse import coo_array
import scipy
import os
import implicit
import pandas as pd
import numpy as np
import datasets
import os
from datetime import datetime

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

/home/jupyter/.local/lib/python3.10/site-packages/implicit/gpu/__init__.py:13: UserWarning: CUDA extension is built, but disabling GPU support because of 'Cuda Error: no CUDA-capable device is detected (/project/./implicit/gpu/utils.h:71)'
  warnings.warn(


In [2]:
dataset = load_from_disk(f"{DATA_PATH}/user_events_20230501")
polars_ds = dataset.to_polars()

In [3]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .filter(pl.col("event_id") == "item_view")
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
)

del polars_ds

In [4]:
train_interactions.shape

(8335353, 18)

In [5]:
event_weights = {
    "item_view": 1,
    "item_like": 1,
    "item_add_to_cart_tap": 1,
    "offer_make": 1,
    "buy_start": 1,
    "buy_comp": 1
}

train_interactions = (
    train_interactions
    .with_columns(pl.col("event_id").apply(lambda x: event_weights[x]).alias("event_weight"))
)

agg_train_interactions = (
    train_interactions
    .groupby("user_id", "item_id")
    .agg(pl.sum("event_weight").alias("interaction_weight"))
)

In [6]:
def df_to_matrix(df, weight=400):
    item2id = {item: i for i, item in enumerate(train_interactions["item_id"].unique())}
    user2id = {user: i for i, user in enumerate(train_interactions["user_id"].unique())}
    id2user = {i: user for i, user in enumerate(train_interactions["user_id"].unique())}
    users_cnt = len(user2id)
    items_cnt = len(item2id)
    pandas_df = df.select("user_id", "item_id", "interaction_weight").unique().to_pandas()
    pandas_df["item_id"] = pandas_df["item_id"].apply(lambda x: item2id[x])
    pandas_df["user_id"] = pandas_df["user_id"].apply(lambda x: user2id[x])
    result = coo_array((pandas_df["interaction_weight"] * weight, (pandas_df["user_id"], pandas_df["item_id"])), shape=(users_cnt, items_cnt))
    return result, item2id, user2id

In [7]:
interaction_matrix, item2id, user2id = df_to_matrix(agg_train_interactions)
sparse_interactions = scipy.sparse.csr_matrix(interaction_matrix)
id2user = {v: k for k, v in user2id.items()}
id2item = {id_: item for item, id_ in item2id.items()}

In [8]:
interaction_matrix.shape

(208093, 4448253)

In [11]:
import json

#with open("item2id_v2.json", "w") as f:
#    json.dump(item2id, f)
    
#with open("user2id_v2.json", "w") as f:
#    json.dump(user2id, f)

In [17]:
TOP_K_VALUES = [10, 50, 400]

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

In [20]:
model = implicit.als.AlternatingLeastSquares(factors=456, iterations=30, regularization=0.001, calculate_training_loss=True, random_state=42)
model.fit(sparse_interactions)
als_recs = model.recommend(range(sparse_interactions.shape[0]), sparse_interactions, N=400, filter_already_liked_items=True)[0]

100%|██████████| 30/30 [02:27<00:00,  4.93s/it, loss=0.000722]


In [26]:
als_recs.shape

(208093, 600)

In [40]:
TOP_K_VALUES = [10, 50, 400]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def get_recs(row):
    if row["user_id"] in user2id:
        als_ids = als_recs[user2id[row["user_id"]]]
        recs = [int(id2item[id_]) for id_ in als_ids]
        return recs
    return []

recs = (
    test_interactions
    .with_columns(
        pl.struct(["user_id"]).apply(get_recs).alias("recs")
    )
)

metrics = (
    recs
    .filter(pl.col("recs").arr.first().is_not_null())
    .with_columns(
        pl.col("future_clicks").apply(len).alias("future_len"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision")
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000")
    )
    .head(5)
)

In [41]:
metrics

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64
0.002839,0.0083,0.025165,0.00361,0.002466,0.001137,0.031035,0.084949,0.200378
